# Mock Photon Propagation - Evaluation

This notebook should give us an overview of the geometric properties of the mock photon propagation. With this we want to evaluate rightful behaviour. To do that we will leave the Olympus way of doing things with event generators and the detector builder. Instead, we will "Mock" our detector to accompany our specific needs!

## Let us import everything first

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go


from ananke.models.detector import Detector
from ananke.models.geometry import Vectors3D
from ananke.models.event import EventRecords, Sources
from ananke.models.collection import Collection
from ananke.schemas.detector import DetectorSchema
from ananke.schemas.event import EventType, SourceType
from ananke.configurations.collection import HDF5StorageConfiguration
from olympus.configuration.photon_propagation import MockPhotonPropagatorConfiguration
from olympus.event_generation.medium import MediumEstimationVariant
from olympus.event_generation.photon_propagation.mock_photons import MockPhotonPropagator


In [ ]:

import plotly.express as px
import pandas as pd
def plot_plotly(x,y,x_title,y_title):
    df = pd.DataFrame({
        'x': x,
        'y': y
    })
    fig = px.line(df, x='x', y='y', labels={'x': x_title, 'y': y_title})
    fig.update_layout(
        autosize=False,
        width=800,
        height=600,
        template='simple_white'
    )
    fig.update_yaxes(rangemode="tozero")
    return fig

## Build Mock Detector

To evaluate the mock photon propagation, we want to build a mock detector. This detector will contain different modules with specific pmt locations and distances. Specifically, we will look at following modules:

* PMT To Cherenkov orientation: Rotate the PMT along the direction of the source (0 / 180)
* PMT To Source rotation: Precession of PMTs against pmt to source direction
* PMT Distances: PMTs at specific distances

To see how we can construct our own detector we can have a look at the Detector DataFrame Schema.

In [ ]:
print(DetectorSchema.to_schema())

First, let's define some constants for the detector:

In [ ]:
dark_noise_rate = 16 * 1e-5  # 1/ns
pmt_cath_area_r = 75e-3 / 2  # m
module_radius = 0.21  # m
efficiency = 0.42 # Christian S. Number
number_of_angles = 36

Okay let's get startet with the module for the cherenkov. For that we will assume that the source is heading along the z-axis. and is positioned on the x-axis. Hence, we will rotate one PMT along the xz-plane starting with the direction of the z-axis.

In [ ]:
spherical_orientations = np.zeros((number_of_angles, 3))

spherical_orientations[:, 0] = 1
module_1_angle_range = np.linspace(0, -2* np.pi, number_of_angles)
spherical_orientations[:, 2] = module_1_angle_range
spherical_orientations_df = pd.DataFrame(spherical_orientations, columns=('norm', 'phi', 'theta'))
cartesian_orientations = Vectors3D.from_spherical(spherical_orientations_df)

module_1_df = cartesian_orientations.get_df_with_prefix('pmt_orientation_')
module_1_df['pmt_id'] = range(number_of_angles)
module_1_df['pmt_efficiency'] = efficiency
module_1_df['pmt_area'] = pmt_cath_area_r
module_1_df['pmt_noise_rate'] = dark_noise_rate
module_1_df['pmt_location_x'] = 0.0
module_1_df['pmt_location_y'] = 0.0
module_1_df['pmt_location_z'] = 0.0
module_1_df['module_id'] = 0
module_1_df['module_radius'] = module_radius
module_1_df['module_location_x'] = 0.0
module_1_df['module_location_y'] = 0.0
module_1_df['module_location_z'] = 0.0
module_1_df['string_id'] = 0
module_1_df['string_location_x'] = 0.0
module_1_df['string_location_y'] = 0.0
module_1_df['string_location_z'] = 0.0

module_1_detector = Detector(df=module_1_df)
module_1_detector.df.head()

Now let's define the precession pmt. We start with the PMT directly facing the source and rotate it around the xy-plane

In [ ]:
spherical_orientations = np.zeros((number_of_angles, 3))

spherical_orientations[:, 0] = 1
module_2_angle_range = np.linspace(0, 2*np.pi, number_of_angles)
spherical_orientations[:, 1] = module_2_angle_range
spherical_orientations[:, 2] = np.pi / 2
spherical_orientations_df = pd.DataFrame(spherical_orientations, columns=('norm', 'phi', 'theta'))
cartesian_orientations = Vectors3D.from_spherical(spherical_orientations_df)

module_2_df = cartesian_orientations.get_df_with_prefix('pmt_orientation_')
module_2_df['pmt_id'] = range(number_of_angles)
module_2_df['pmt_efficiency'] = efficiency
module_2_df['pmt_area'] = pmt_cath_area_r
module_2_df['pmt_noise_rate'] = dark_noise_rate
module_2_df['pmt_location_x'] = 0.0
module_2_df['pmt_location_y'] = 0.0
module_2_df['pmt_location_z'] = 0.0
module_2_df['module_id'] = 1
module_2_df['module_radius'] = module_radius
module_2_df['module_location_x'] = 0.0
module_2_df['module_location_y'] = 0.0
module_2_df['module_location_z'] = 0.0
module_2_df['string_id'] = 0
module_2_df['string_location_x'] = 0.0
module_2_df['string_location_y'] = 0.0
module_2_df['string_location_z'] = 0.0

module_2_detector = Detector(df=module_2_df)
module_2_detector.df.head()

Now, let's mix the previous cases. We start with a theta of 45° and rotate phi 360°

In [ ]:
spherical_orientations = np.zeros((number_of_angles, 3))

spherical_orientations[:, 0] = 1
module_3_angle_range = np.linspace(0, 2*np.pi, number_of_angles)
spherical_orientations[:, 1] = module_3_angle_range
spherical_orientations[:, 2] = np.pi / 4
spherical_orientations_df = pd.DataFrame(spherical_orientations, columns=('norm', 'phi', 'theta'))
cartesian_orientations = Vectors3D.from_spherical(spherical_orientations_df)

module_3_df = cartesian_orientations.get_df_with_prefix('pmt_orientation_')
module_3_df['pmt_id'] = range(number_of_angles)
module_3_df['pmt_efficiency'] = efficiency
module_3_df['pmt_area'] = pmt_cath_area_r
module_3_df['pmt_noise_rate'] = dark_noise_rate
module_3_df['pmt_location_x'] = 0.0
module_3_df['pmt_location_y'] = 0.0
module_3_df['pmt_location_z'] = 0.0
module_3_df['module_id'] = 2
module_3_df['module_radius'] = module_radius
module_3_df['module_location_x'] = 0.0
module_3_df['module_location_y'] = 0.0
module_3_df['module_location_z'] = 0.0
module_3_df['string_id'] = 0
module_3_df['string_location_x'] = 0.0
module_3_df['string_location_y'] = 0.0
module_3_df['string_location_z'] = 0.0

module_3_detector = Detector(df=module_3_df)
module_3_detector.df.head()

Last, but not least we are varying the distance of the PMTs with the PMTs heading directly towards the source on the x-axis facing the z-direction.

In [ ]:
distance_between_pmts = 5
maximum_distance = 100
number_of_steps = int(maximum_distance / distance_between_pmts)
module_4_range = np.arange(0, maximum_distance, distance_between_pmts)
cartesian_locations = np.zeros((number_of_steps, 3))
cartesian_locations[:, 0] = module_4_range
cartesian_locations = Vectors3D.from_numpy(cartesian_locations)

module_4_df = cartesian_locations.get_df_with_prefix('pmt_location_')
module_4_df['pmt_id'] = range(number_of_steps)
module_4_df['pmt_efficiency'] = efficiency
module_4_df['pmt_area'] = pmt_cath_area_r
module_4_df['pmt_noise_rate'] = dark_noise_rate
module_4_df['pmt_orientation_x'] = -1.0
module_4_df['pmt_orientation_y'] = 0.0
module_4_df['pmt_orientation_z'] = 0.0
module_4_df['module_id'] = 3
module_4_df['module_radius'] = module_radius
module_4_df['module_location_x'] = 0.0
module_4_df['module_location_y'] = 0.0
module_4_df['module_location_z'] = 0.0
module_4_df['string_id'] = 0
module_4_df['string_location_x'] = 0.0
module_4_df['string_location_y'] = 0.0
module_4_df['string_location_z'] = 0.0

module_4_detector = Detector(df=module_4_df)
module_4_detector.df.head()

Let's complete our detector:

In [ ]:
detector = Detector.concat([
    module_1_detector,
    module_2_detector,
    module_3_detector,
    module_4_detector,
])
len(detector.df.index)

## Build Mock Event and Sources

After the detector, we only need to create an event and its sources. Here as well to be able to evaluate the cherenkov dependency we create events and sources starting in y-axis direction and then rotating down the xz-plane

In [ ]:
spherical_orientations = np.zeros((number_of_angles * 4, 3))

spherical_orientations[:, 0] = 1
source_angle_range = np.linspace(0, 2* np.pi, number_of_angles * 4)
spherical_orientations[:, 2] = source_angle_range
spherical_orientations_df = pd.DataFrame(spherical_orientations, columns=('norm', 'phi', 'theta'))
cartesian_orientations = Vectors3D.from_spherical(spherical_orientations_df)

events_df = cartesian_orientations.get_df_with_prefix('orientation_')
events_df['record_id'] = range(number_of_angles * 4)
events_df['time'] = 0.0
events_df['location_x'] = -5.0
events_df['location_y'] = 0.0
events_df['location_z'] = 0.0

sources_df = events_df.copy()

events_df['energy'] = 10000
events_df['length'] = 3000
events_df['particle_id'] = 11
events_df['type'] = EventType.CASCADE.value

sources_df['type'] = SourceType.ISOTROPIC.value
sources_df['number_of_photons'] = 10000000

event_records = EventRecords(df=events_df)
sources = Sources(df=sources_df)

event_records.df.head()

In [ ]:
sources.df.head()

## Let's propagate

Now we have the detector, the events and the sources. Time to propagate:

In [ ]:
configuration = HDF5StorageConfiguration(data_path='data/mock_photon_evaluation.h5', read_only=False)
configuration

In [ ]:
collection = Collection(configuration)

with collection:
    collection.storage.set_detector(detector)
    collection.storage.set_records(event_records)
    collection.storage.set_sources(sources)

In [ ]:
configuration = MockPhotonPropagatorConfiguration(
        resolution=18000,
        medium=MediumEstimationVariant.PONE_OPTIMISTIC
)
photon_propagator = MockPhotonPropagator(
    detector=detector,
    configuration=configuration
)

with collection:
    photon_propagator.propagate(collection)

In [ ]:
with collection:
    all_hits = collection.storage.get_hits()
    hits = collection.storage.get_hits(record_ids=20)
hits_indexed = hits.df.set_index(['string_id', 'module_id', 'pmt_id'])

In [ ]:
hits.df.loc[hits.df['record_id'] ==0.0]['time']
len(hits.df)

Faszinating. Look how many hits we generated. More interestingly, we can now evaluate the properties of the mock photon propatation.

## Evaluation of the Mock Photon Propagation

We want to evaluate the following cases

1. What happens when the PMT rotates along the cherenkov axis?
2. What happens when the PMT rotates perpendicular to the cherenkov axis?
3. What happens when the PMT rotates along both axis at an 45° angle?
4. What happens when the PMT moves further away from the source?
5. What happens when the source direction roates along the cherenkov axis?

First we will define a helper function to draw beautiful graphs. For the first three points we let the source be shining along the x axis only

In [ ]:
def plot(x, y, title, x_label, show=True):
    fig, ax = plt.subplots()
    ax.plot(x, y)
    ax.set_xlabel(x_label)
    ax.set_ylabel('Photon Count')
    ax.set_title(title)
    if show:
        plt.show()
    

### What happens when the PMT rotates along the cherenkov axis?

In this case the angle between the PMT and the source changes from the PMT taking the same direction as the Source to facing the source directly and continuing on that plane until the PMT faces the opposite direction.

In [ ]:
grouped_hits = hits_indexed.groupby(level=[0,1,2]).count()

In [ ]:
module_1_hits = np.zeros_like(module_1_angle_range)
module_1_pd_hits = grouped_hits.loc[0,0,:]['type']
module_1_hits[module_1_pd_hits.index.values] = module_1_pd_hits

fig = plot_plotly(module_1_angle_range * 180 / np.pi, module_1_hits, 'PMT angles [°]', 'Hit count')
fig.write_image('data/module_1.png', scale=2)
#plot(module_1_angle_range * 180 / np.pi, module_1_hits, 'Behaviour for PMT rotation along cherenkov axis', 'PMT Angles [°]')
fig.show()

### What happens when the PMT rotates perpendicular to the cherenkov axis?

Next up we check the same thing. Only that the the PMT is now facing the source directly at the beginning and moves with an angle on the plane having the source orientation as its normal vector.

In [ ]:
module_2_hits = np.zeros_like(module_2_angle_range)
module_2_pd_hits = grouped_hits.loc[0,1,:]['type']
module_2_hits[module_2_pd_hits.index.values] = module_2_pd_hits

fig = plot_plotly(module_2_angle_range * 180 / np.pi, module_2_hits, 'PMT angles [°]', 'Hit count')
fig.write_image('data/module_2.png', scale=2)
fig.show()
#plot(module_2_angle_range * 180 / np.pi, module_2_hits, 'Behaviour for PMT rotation perpendicular to cherenkov axis', 'PMT Angles [°]')

### What happens when the PMT rotates along both axis at an 45° angle?

This time, we combine the previous two methods having the PMT at an angle of 45° towards the plane having the orientation of the source as a normal.


In [ ]:
module_3_hits = np.zeros_like(module_3_angle_range)
module_3_pd_hits = grouped_hits.loc[0,2,:]['type']
module_3_hits[module_3_pd_hits.index.values] = module_3_pd_hits

fig = plot_plotly(module_3_angle_range * 180 / np.pi, module_3_hits, 'PMT angles [°]', 'Hit count')
fig.write_image('data/module_3.png', scale=2)
fig.show()
#plot(module_3_angle_range * 180 / np.pi, module_3_hits, 'Behaviour for PMT rotation at 45°', 'PMT Angles [°]')

### What happens if the source moves further away?

This time the PMT faces the source directly and moves away slowly.

In [ ]:
module_4_hits = np.zeros_like(module_4_range)
module_4_pd_hits = grouped_hits.loc[0,3,:]['type']
module_4_hits[module_4_pd_hits.index.values] = module_4_pd_hits

fig = plot_plotly(module_4_range, module_4_hits, 'PMT Distance [m]', 'Hit count')
fig.write_image('data/distance.png', scale=2)
fig.show()

### What happens if the Source rotates towards PMT?

Now we take the closest PMT of the last experiment with the PMT distance and evaluate what happens when the source first goes perpendicular to the PMT and then rotates until it faces it and back until 360° are reached

In [ ]:
with collection:
    records = collection.storage.get_records()
source_hits_df = all_hits.df[(all_hits.df['string_id']==0) & (all_hits.df['module_id']==3) & (all_hits.df['pmt_id']==0)]
source_hits_df = source_hits_df.set_index('record_id')

grouped_source_hits = source_hits_df.groupby(level=[0]).count()['type']

for i in range(len(source_angle_range)):
    if i not in grouped_source_hits.index:
        grouped_source_hits.at[i]= 0
len_of_sources = len(grouped_source_hits)
a_quarter = int(len_of_sources/4)

fig = plot_plotly(source_angle_range[a_quarter:a_quarter*3] * 180 / np.pi - 90, grouped_source_hits[a_quarter:a_quarter*3], 'Source angle [°]', 'Hit count')
fig.add_vline(x=40.72385005, line_width=3, line_dash="dash", line_color="green", annotation=dict(text='Cherenkov angle'))
fig.write_image('data/sources.png', scale=2)
fig.show()

In [ ]:
first_record_hits = all_hits.df[(all_hits.df['string_id']==0) & (all_hits.df['module_id']==3) & (all_hits.df['pmt_id']==0) & (all_hits.df['record_id']==0)][['time']]
fig = px.histogram(first_record_hits['time'], nbins=60)
fig.write_image('data/hit_times_1.png', scale=2)
fig.update_layout(
    autosize=False,
    width=800,
    height=600,
    template='simple_white'
)
fig.update_yaxes(rangemode="tozero")
fig.update_xaxes(rangemode="tozero")
fig.show()

In [ ]:
first_record_hits = all_hits.df[(all_hits.df['string_id']==0) & (all_hits.df['module_id']==3) & (all_hits.df['pmt_id']==0) & ((all_hits.df['record_id']==3) | (all_hits.df['record_id']==6) | (all_hits.df['record_id']==9))]
fig = px.histogram(first_record_hits, x='time', color='record_id', nbins=60, labels={'x': 'Hit time [ns]'})
fig.update_layout(
    showlegend=False,
    autosize=False,
    width=800,
    height=600,
    template='simple_white'
)
fig.update_yaxes(rangemode="tozero", title_text='Hit count')
fig.update_xaxes(title_text='Time [ns]', min=20)
fig.write_image('data/hit_times_2.png', scale=2)
fig.show()

In [ ]:
all_record_hits = all_hits.df[(all_hits.df['string_id']==0) & (all_hits.df['module_id']==3) & (all_hits.df['pmt_id']==0)]
fig = px.density_heatmap(all_record_hits, x='time', y='record_id', histnorm='probability')
fig.show()

In [ ]:
px.data.tips()